# Lab01 — Example: ba kỹ thuật của bài này trên dữ liệu tí hon

(Cập nhật lần cuối 25/09/2026)

File này **không phải bài tập, không nộp, không có điểm**.

1. Vì sao `line.split('\t')` là **sai** với file TSV có quoting.
2. Cách đo bộ nhớ đỉnh (peak) bằng `tracemalloc`, và chênh lệch giữa *đọc cả file* và *đọc từng dòng*.
3. Hình dạng của `git bisect run` trên một thùng chứa 5 commit.

Bạn chạy `Kernel` → `Restart & Run All` rồi đọc output. Không có chỗ nào phải điền.

---

## 1. Vì sao `split('\t')` là sai

Mình ghi ra một file tí hon 3 cột, 3 record. Record thứ hai có một giá trị **chứa tab**, record thứ ba có một giá trị **chứa ký tự xuống dòng** — cả hai đều được bọc trong dấu nháy kép theo đúng quy ước của đề.

In [ ]:
noi_dung = (
    'Id\tSubject\tFile\n'
    'aaa\tSua loi nho\ta.py\n'
    'bbb\t"Them cot\tThoi luong"\tb.py\n'
    'ccc\t"Dong mot\nDong hai"\tc.py\n'
)
with open('vidu.tsv', 'w', encoding='utf-8') as f:
    f.write(noi_dung)

print(repr(noi_dung))

'Id\tSubject\tFile\naaa\tSua loi nho\ta.py\nbbb\t"Them cot\tThoi luong"\tb.py\nccc\t"Dong mot\nDong hai"\tc.py\n'


File chỉ có 3 record, nhưng đếm số dòng của file thì ra **5** — vì một giá trị chứa ký tự xuống dòng. Đây là lý do `my_tsv_reader` phải đọc theo **ký tự**, chứ không đọc theo dòng:

In [ ]:
with open('vidu.tsv', encoding='utf-8') as f:
    cac_dong = f.read().splitlines()

print('so dong cua file :', len(cac_dong))
print('so record thuc su :', 3)
print()
for i, dong in enumerate(cac_dong):
    print(i, repr(dong))

so dong cua file : 5
so record thuc su : 3

0 'Id\tSubject\tFile'
1 'aaa\tSua loi nho\ta.py'
2 'bbb\t"Them cot\tThoi luong"\tb.py'
3 'ccc\t"Dong mot'
4 'Dong hai"\tc.py'


Và đây là kết quả khi cứ `split('\t')` từng dòng — bạn để ý record `bbb` bị tách thành **4** giá trị thay vì 3, còn record `ccc` thì bị vỡ làm hai dòng:

In [ ]:
for dong in cac_dong:
    print(len(dong.split('\t')), dong.split('\t'))

3 ['Id', 'Subject', 'File']
3 ['aaa', 'Sua loi nho', 'a.py']
4 ['bbb', '"Them cot', 'Thoi luong"', 'b.py']
2 ['ccc', '"Dong mot']
2 ['Dong hai"', 'c.py']


Kết quả **đúng** phải là:

```
['aaa', 'Sua loi nho', 'a.py']
['bbb', 'Them cot\tThoi luong', 'b.py']
['ccc', 'Dong mot\nDong hai', 'c.py']
```

Nghĩa là: dấu nháy kép ở đầu một giá trị **bật** chế độ quoting, và trong lúc quoting thì tab và ký tự xuống dòng là **dữ liệu**, không phải dấu ngăn. Việc của bạn ở phần A là viết ra thứ làm đúng chuyện đó.

---

## 2. Đo bộ nhớ đỉnh bằng `tracemalloc`

Khuôn đo luôn là bốn dòng này:

```python
tracemalloc.start()
... đoạn code cần đo ...
peak = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
```

`get_traced_memory()` trả về `(bộ nhớ đang dùng, bộ nhớ đỉnh)` — ta lấy phần tử thứ hai. Dưới đây mình sinh một file 5 MB rồi đếm số dòng của nó bằng hai cách, đo cả hai.

In [ ]:
import os
import time
import tracemalloc

with open('vidu_lon.txt', 'w', encoding='utf-8') as f:
    for i in range(100000):
        f.write(f'dong so {i}, mot chut chu cho day day mot chut nua\n')
print('kich thuoc file: %.1f MB' % (os.path.getsize('vidu_lon.txt') / 1024 ** 2))


def dem_dong_doc_ca_file(path):
    with open(path, encoding='utf-8') as f:
        return len(f.read().split('\n')) - 1


def dem_dong_tung_dong(path):
    so = 0
    with open(path, encoding='utf-8') as f:
        for _ in f:
            so += 1
    return so


for ten, ham in (('doc ca file', dem_dong_doc_ca_file),
                 ('doc tung dong', dem_dong_tung_dong)):
    tracemalloc.start()
    t0 = time.perf_counter()
    ket_qua = ham('vidu_lon.txt')
    giay = time.perf_counter() - t0
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    print(f'{ten:15} {ket_qua} dong | {giay:5.2f}s | peak {peak / 1024 ** 2:8.2f} MiB')

kich thuoc file: 5.0 MB
doc ca file     100000 dong |  0.07s | peak    15.44 MiB
doc tung dong   100000 dong |  0.09s | peak     0.02 MiB


Hai cách cho **cùng một kết quả**, nhưng bộ nhớ đỉnh chênh nhau vài nghìn lần. Ở phần B của bài, cell `# BENCHMARK` sẽ đo đúng kiểu này trên file 57 MB, và đề đặt ngưỡng theo **tỉ lệ** (`peak_naive / peak_gen`) chứ không theo số MB tuyệt đối — vì máy mỗi bạn một khác.

---

## 3. `git bisect run` trông như thế nào

Mình dựng một thùng chứa 5 commit, trong đó commit thứ tư làm `kiem_tra.py` fail, rồi để `git bisect` tự tìm.

In [ ]:
import shutil
import subprocess

shutil.rmtree('vidu-repo', ignore_errors=True)
os.makedirs('vidu-repo')

def g(*args):
    return subprocess.run(['git', '-C', 'vidu-repo', *args],
                          capture_output=True, text=True).stdout

g('init', '--quiet', '--initial-branch=main')
g('config', 'user.name', 'Vi du')
g('config', 'user.email', 'vidu@example.com')

with open('vidu-repo/kiem_tra.py', 'w') as f:
    f.write('import cong\nassert cong.cong(2, 3) == 5\n')

cac_ban = ['def cong(a, b):\n    return a + b\n',            # commit 1 - tot
           'def cong(a, b):\n    return a + b  # ok\n',      # commit 2 - tot
           'def cong(a, b):\n    return (a + b)\n',          # commit 3 - tot
           'def cong(a, b):\n    return a - b\n',            # commit 4 - HONG
           'def cong(a, b):\n    return a - b  # refactor\n']  # commit 5 - hong
for i, ban in enumerate(cac_ban, start=1):
    with open('vidu-repo/cong.py', 'w') as f:
        f.write(ban)
    g('add', '-A')
    g('commit', '--quiet', '-m', f'ban thu {i}')

print(g('log', '--oneline'))

8d08f1a ban thu 5
219be94 ban thu 4
029fea7 ban thu 3
33b83b3 ban thu 2
9a81553 ban thu 1



In [ ]:
commit_dau = g('rev-list', '--max-parents=0', 'HEAD').split()[0]

g('bisect', 'start')
g('bisect', 'bad')
g('bisect', 'good', commit_dau)
ket_qua = subprocess.run(['git', '-C', 'vidu-repo', 'bisect', 'run',
                          'python3', 'kiem_tra.py'],
                         capture_output=True, text=True).stdout
g('bisect', 'reset')
print(ket_qua[-600:])

running 'python3' 'kiem_tra.py'
Bisecting: 0 revisions left to test after this (roughly 0 steps)
[219be942267bd8f6b052c6377ee469f59083e6c8] ban thu 4
running 'python3' 'kiem_tra.py'
219be942267bd8f6b052c6377ee469f59083e6c8 is the first bad commit
commit 219be942267bd8f6b052c6377ee469f59083e6c8
Author: Vi du <vidu@example.com>
Date:   Sun Sep 20 00:51:31 2026 +0700

    ban thu 4

 cong.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
bisect found first bad commit



Dòng có chữ *"is the first bad commit"* là câu trả lời — và bạn để ý: `git bisect` chỉ phải thử **2 commit** thay vì 5. Với thùng chứa 2000 commit thì nó chỉ thử khoảng 11 commit. Đó là lý do bài bắt bạn dùng `git bisect` thay vì mở từng commit ra xem.

<font color=gray>Dọn rác: các file `vidu.tsv`, `vidu_lon.txt`, `vidu-repo/` do file này sinh ra, bạn xoá lúc nào cũng được.</font>